# Collect results
Uncomment the benchmarks and methods you want. The run cell evaluates **every selected method on every selected benchmark**.

In Colab, select a GPU runtime for GPU methods and run Setup; it clones the public repository automatically. Locally, open this notebook from the checkout.

Setup keeps the installed NumPy/SciPy pair and CUDA PyTorch, so a fresh Colab session should not need a restart. If an earlier Setup already replaced loaded NumPy/SciPy, choose **Runtime → Restart session** once, then rerun Setup.


In [ ]:
#@title Setup
import os, runpy, subprocess
from pathlib import Path

setup = next((p for p in [Path('tools/colab.py'), Path('../tools/colab.py'), Path('decision-rerank/tools/colab.py')] if p.exists()), None)
if setup is None:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ellimilial/decision-rerank.git', 'decision-rerank'],
                   check=True, env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    setup = Path('decision-rerank/tools/colab.py')
setup_notebook = runpy.run_path(str(setup))['setup_notebook']
setup_notebook()
# setup_notebook(linear_kernels=True)  # Use instead of the line above for the optional L4/A100 FLA trial.


In [ ]:
from decision_rerank.collection import benchmark, method, collect_results, download_results


In [ ]:
# Optional Hugging Face token. Skip this cell or press Enter to keep the current setting.
import os
from getpass import getpass

token = getpass("HF_TOKEN (Enter to skip): ").strip()
if token:
    os.environ["HF_TOKEN"] = token
del token


## Benchmarks
The default smoke test is **one query from each benchmark**, with all 100 candidates per query. Data preparation and model downloads can still take time.

For a throughput check, replace the selections with the commented **two-query NFCorpus** entry. The first query includes model loading and kernel compilation; the second is the useful warmed-up estimate. Full panels and full datasets are separate alternatives, not additional copies of the same benchmark.


In [ ]:
BENCHMARKS = [
    benchmark('nfcorpus_dev', panel='nfcorpus_pilot', limit=1),
    benchmark('bioasq13b_curated', panel='bioasq', limit=1),
    benchmark('curev1_en', panel='cure', limit=1),

    # Throughput trial: use this alone, replacing the three smoke selections above.
    # benchmark('nfcorpus_dev', panel='nfcorpus_pilot', limit=2),

    # Larger panels: replace the smoke selections above when throughput is acceptable.
    # benchmark('nfcorpus_dev', panel='nfcorpus_pilot'),          # 20-query development pilot
    # benchmark('bioasq13b_curated', panel='bioasq'),               # 30-query panel
    # benchmark('curev1_en', panel='cure'),                        # 30-query panel

    # Full datasets: alternatives to the corresponding panels.
    # benchmark('bioasq13b_curated'),                              # 340 queries
    # benchmark('curev1_en'),                                     # 2,000 queries
    # benchmark('nfcorpus_dev'),                                  # 324 dev queries
]


## Methods
Uncomment any combination. Each local model runs across all selected benchmarks, then unloads. Start with BM25, MedCPT, Qwen3 reranker, JEV and Luna; measure the decision models before larger panels.

**Optional Qwen3.5 FLA trial (BioDecision, Kev, qwen35-base):** select `setup_notebook(linear_kernels=True)` instead of ordinary Setup in a fresh L4/A100 session. Current Triton excludes T4. FLA is pinned in the lockfile; existing CUDA torch/Triton are retained. We do not compile `causal-conv1d`, so a partial fast-path warning can remain: check the printed **delta_rule=fla**. Active paths/versions are saved in `run.json`.

Compare the same two-query selection and one model in separate fresh sessions/output folders, e.g. `results-torch` and `results-fla`. Compare second-query times and rankings; the first includes loading/compilation. Kernel metadata is new: use a new output folder for Qwen3.5-based runs with this version. GPU speed and numerical agreement remain to be validated. [Kernel requirements and details](https://github.com/ellimilial/decision-rerank#decision-rerank).

Missing weights download automatically. Hosted methods make paid calls. Kev defaults to FP16/batch size 1; fit needs validation. Sol ranks all 100 candidates in one response; invalid rankings fail without repair.


In [ ]:
METHODS = [
    # GPU rerankers: try these first, roughly larger to smaller.
    # method('qwen3-4b'),        # Qwen/Qwen3-Reranker-4B; different architecture from Qwen3.5
    # method('jina-v35'),        # 0.6B: joint list; bounded attention can still be slow
    # method('medcpt'),          # Biomedical cross-encoder baseline

    # GPU decision models: measure warmed-up throughput before larger panels.
    # method('biodecision'),     # 4B: lighteternal/biodecision-v2-4b
    # method('kev-4b'),          # 4B: direct GPU loading, batch size 1
    # method('qwen35-base'),     # 4B: general LLM control; lower priority

    # Lexical controls (CPU)
    # method('tfidf-cpu'),
    method('bm25'),

    # Hosted: no local GPU; paid API calls
    # method('jev'),
    # method('openai-decisions'),

    # Sol: one-shot full ranking
    # method('sol-listwise-low'),   # 1 request per query
    # method('sol-listwise-high'),  # Optional extra reasoning
]


## Run
Selected hosted methods check their keys **before any model runs**:

- JEV, OpenAI Luna Decisions and GPT-6.1 Sol: `OPENROUTER_API_KEY`

Existing environment variables or Colab Secrets are reused. Otherwise, enter the key in the hidden prompt. Keys stay in runtime memory and are not written into the notebook or result files. CPU/GPU-only selections need no keys.

Results are saved after each query. Reuse the output folder to resume.


In [ ]:
archive = collect_results(BENCHMARKS, METHODS, output='results')


In [ ]:
download_results(archive)
